# pipe_catedra/00 — Resumen de experimentos

No genera ningun dataset ni modelo: solo lee lo que ya esta en `DIR_OUT` (el
bucket) y arma un inventario, para saber que corriste sin tener que
acordarte de memoria.

Se puede correr en cualquier momento, incluso sin haber corrido nada todavia
(muestra todo vacio) y las veces que haga falta (no modifica nada).

Responsabilidades:
- Listar los archivos que hay en `DIR_OUT` (que datasets/inferencias/predicciones ya existen)
- Leer todos los `z303_hiper_*.json` y armar una tabla ordenada por mejor metrica
- Marcar que combinaciones de `modo_agrupacion` x `tipo_target` x `metodo_escalado`
  ya se corrieron en 03_Optuna y cuales faltan


## 0) Setup


In [ ]:
import json
import os
from pathlib import Path


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_OUT = BUCKET / "exp_pipe_catedra"
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"salida : {DIR_OUT}")


## 1) Inventario de archivos


In [ ]:
print(f'=== Archivos en {DIR_OUT} ===')
archivos = sorted(DIR_OUT.iterdir()) if DIR_OUT.exists() else []
if not archivos:
    print('  (vacio -- todavia no corriste nada)')
for f in archivos:
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name:55s} {size_mb:8.1f} MB')


## 2) Experimentos de 03_Optuna, ordenados por mejor metrica


In [ ]:
jsons = sorted(DIR_OUT.glob('z303_hiper_*.json'))

filas = []
for jf in jsons:
    cfg = json.loads(jf.read_text())
    filas.append({
        'archivo':         jf.name,
        'experimento':     cfg.get('experimento'),
        'modo':            cfg.get('modo_agrupacion'),
        'tipo_target':     cfg.get('tipo_target'),
        'metodo_escalado': cfg.get('metodo_escalado'),
        'esquema_val':     cfg.get('esquema_val'),
        'objective_lgbm':  cfg.get('objective_lgbm'),
        'metrica':         cfg.get('metrica'),
        'mejor_valor':     cfg.get('mejor_valor'),
        'n_features':      len(cfg.get('features', [])),
    })

filas.sort(key=lambda f: (f['mejor_valor'] is None, f['mejor_valor']))

if not filas:
    print('(No hay experimentos de 03_Optuna guardados todavia.)')
else:
    header = (f"{'experimento':12s} {'modo':16s} {'target':7s} {'escalado':9s} "
              f"{'val':10s} {'obj':13s} {'metrica':7s} {'valor':8s} {'feats':5s}")
    print(header)
    print('-' * len(header))
    for f in filas:
        valor = f"{f['mejor_valor']:.4f}" if f['mejor_valor'] is not None else '?'
        print(f"{str(f['experimento']):12s} {str(f['modo']):16s} {str(f['tipo_target']):7s} "
              f"{str(f['metodo_escalado']):9s} {str(f['esquema_val']):10s} "
              f"{str(f['objective_lgbm']):13s} {str(f['metrica']):7s} {valor:8s} {f['n_features']:5d}")


## 3) Combinaciones probadas vs. pendientes

Grilla de `modo_agrupacion` x `tipo_target` x `metodo_escalado` (los ejes que
mas cambian el resultado). Otras palancas (`objective_lgbm`, `regularizacion`,
`decay_recencia`, `esquema_val`) quedan afuera de la grilla -- se ven en la
tabla de arriba por cada experimento puntual.


In [ ]:
modos     = ['producto', 'cliente_producto']
targets   = ['nivel', 'delta']
escalados = [None, 'media', 'zscore', 'rango']
hechas = {(f['modo'], f['tipo_target'], f['metodo_escalado']) for f in filas}

header = f"{'modo':16s} {'tipo_target':11s} {'metodo_escalado':15s} estado"
print(header)
print('-' * len(header))
for modo in modos:
    for target in targets:
        for esc in escalados:
            estado = 'ya corrida' if (modo, target, esc) in hechas else 'pendiente'
            print(f'{modo:16s} {target:11s} {str(esc):15s} {estado}')
